# Aberfoyle 2008, Block 21: understand one metrics file

This notebook investigates one CSV before any modelling or multi-year comparison. The workflow is deliberately ordered:

1. establish what one row and one column represent;
2. check structure and data quality without changing the raw data;
3. validate relationships implied by the column names;
4. examine distributions, zero values, spatial patterns, and relationships;
5. record conclusions and unresolved metadata questions.

A column name can suggest a meaning, but it cannot establish the exact LiDAR processing definition. Keep interpretations tentative until the data provider's metric documentation is available.

## 1. Setup and load the file

In [7]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

CSV_RELATIVE_PATH = Path("lidar_data_29jun/Aberfoyle2008/Aberfoyle_2008_Metrics_Block21.csv")
DATA_PATH = next(
    (folder / CSV_RELATIVE_PATH for folder in [Path.cwd(), Path.cwd().parent] if (folder / CSV_RELATIVE_PATH).exists()),
    None,
)
if DATA_PATH is None:
    raise FileNotFoundError(f"Could not find {CSV_RELATIVE_PATH}. Current folder: {Path.cwd()}")

raw_df = pd.read_csv(DATA_PATH)
df = raw_df.copy()  # raw_df remains unchanged during exploration
print(f"Loaded: {DATA_PATH.resolve()}")

Loaded: lidar_data_29jun/Aberfoyle2008/Aberfoyle_2008_Metrics_Block21.csv


In [9]:
print(f"Rows: {len(df):,}")
print(f"Columns: {df.shape[1]}")
df.head()

Rows: 71,996
Columns: 62


,X,Y,PolygonFID,CanopyCover,GapFraction,LAI,elev_aad_z,elev_canopy_relief_ratio,elev_AIH_1st,elev_AIH_5th,elev_AIH_10th,elev_AIH_20th,elev_AIH_25th,elev_AIH_30th,elev_AIH_40th,elev_AIH_50th,elev_AIH_60th,elev_AIH_70th,elev_AIH_75th,elev_AIH_80th,elev_AIH_90th,elev_AIH_95th,elev_AIH_99th,elev_AIH_IQ,elev_curt_mean_cube,elev_cv_z,elev_IQ,elev_kurtosis,elev_madmedian,elev_max,elev_min,elev_mean,elev_median_z,elev_percentile_1st,elev_percentile_5th,elev_percentile_10th,elev_percentile_20th,elev_percentile_25th,elev_percentile_30th,elev_percentile_40th,elev_percentile_50th,elev_percentile_60th,elev_percentile_70th,elev_percentile_75th,elev_percentile_80th,elev_percentile_90th,elev_percentile_95th,elev_percentile_99th,elev_skewness,elev_sqrt_mean_sq,elev_stddev,elev_variance,density_metrics[0],density_metrics[1],density_metrics[2],density_metrics[3],density_metrics[4],density_metrics[5],density_metrics[6],density_metrics[7],density_metrics[8],density_metrics[9]
248077.406,705230.498,0,0.805139,0.194861,0.781303,3.842699,0.617803,2.557115,7.001604,10.783807,14.399079,15.210015,15.847497,17.270166,18.435190,19.444284,20.577349,21.122156,21.663195,23.369715,24.134943,26.375944,5.079414,18.838606,0.284908,5.912141,3.913919,2.923332,27.247847,2.063806,17.622583,18.427095,6.431852,11.724984,14.016000,15.880059,16.789347,17.382692,18.487467,19.438044,20.469080,21.342684,21.868761,22.465807,23.734673,24.673292,26.528616,-0.947529,18.323865,5.020817,25.208608,0.031028,0.020390,0.023936,0.052305,0.086879,0.177305,0.234929,0.213652,0.125000,0.033688,NaN
248097.406,705230.498,1,0.758017,0.241983,0.677838,2.860214,0.675393,3.210779,8.870881,13.270883,15.698304,16.253206,16.840998,17.698366,18.367708,19.110651,19.949678,20.242159,20.665323,21.574717,22.414713,24.560629,3.493584,18.450224,0.229563,3.988953,6.258910,1.983570,25.105734,2.192056,17.667799,18.364580,6.765630,13.367181,14.897058,16.596510,17.157698,17.590540,18.215631,18.855730,19.569887,20.291109,20.651281,21.056139,21.943911,22.728809,24.605730,-1.594478,18.127362,4.055875,16.450119,0.024038,0.017308,0.009615,0.017308,0.048077,0.108654,0.255769,0.300962,0.180769,0.036538,NaN
248077.406,705250.498,2,0.662420,0.337580,0.518786,3.408595,0.660559,3.729311,6.747758,10.974526,14.744981,15.639344,16.435305,17.606203,18.542639,19.307905,20.195364,20.504965,20.939611,22.185118,22.819174,23.903868,3.824692,18.470702,0.259381,4.865621,4.328603,2.287941,25.303869,2.251581,17.478981,18.542639,6.022737,11.964936,14.304559,16.315725,17.123899,17.592325,18.501900,19.226698,19.916700,20.544394,20.948591,21.514227,22.523108,23.155203,24.026335,-1.249482,18.057389,4.533709,20.554517,0.021368,0.028846,0.029915,0.027778,0.055556,0.118590,0.199786,0.295940,0.180556,0.041667,NaN
248097.406,705250.498,3,0.687845,0.312155,0.556193,3.009122,0.664477,3.373079,8.898782,12.284471,13.966821,14.862412,15.498780,16.570566,17.395197,18.302866,19.180792,19.625216,19.997591,21.352461,22.238562,23.665899,4.161427,17.686846,0.234091,4.762803,4.525903,2.355949,24.310255,2.152460,16.875813,17.393805,7.706941,12.229978,13.357739,15.314187,15.899781,16.403963,17.326086,18.059084,19.017416,19.729809,20.061209,20.606745,21.677557,22.583605,23.761946,-1.028408,17.332032,3.950476,15.606263,0.015060,0.011044,0.022088,0.025100,0.083333,0.134538,0.241968,0.251004,0.162651,0.051205,NaN
248117.406,705250.498,4,0.713167,0.286833,0.596608,2.294197,0.656566,3.613229,8.403092,12.181527,14.034112,14.429826,14.848458,15.483415,15.957063,16.538538,17.103868,17.500711,17.781572,18.961145,19.731188,21.285196,2.759274,16.149067,0.216471,3.070886,6.623762,1.534448,22.515200,2.173418,15.529133,15.956941,6.005479,12.172013,13.527600,14.624104,15.001138,15.335599,15.881188,16.354387,16.901533,17.533525,17.760412,18.075243,19.304750,20.129900,21.698641,-1.576211,15.888814,3.361613,11.300445,0.023952,0.016966,0.008982,0.018962,0.039920,0.134731,0.331337,0.290419,0.106786,0.027944,NaN


### First interpretation of the observational unit

The `X` and `Y` fields appear to locate a spatial unit, and `PolygonFID` appears to identify it. Each remaining field is a metric calculated for that unit. We test uniqueness below. The coordinate reference system, polygon size, return filters, height normalisation, and metric definitions must still be confirmed from metadata.

## 2. Inventory the columns

In [10]:
spatial_cols = [c for c in ["X", "Y", "PolygonFID"] if c in df]
canopy_cols = [c for c in ["CanopyCover", "GapFraction", "LAI"] if c in df]
elevation_cols = [c for c in df if c.startswith("elev_")]
density_cols = [c for c in df if c.startswith("density_metrics")]
known_cols = set(spatial_cols + canopy_cols + elevation_cols + density_cols)
other_cols = [c for c in df if c not in known_cols]

column_groups = pd.DataFrame({
    "group": ["spatial / identifier", "canopy", "elevation", "density", "unclassified"],
    "number_of_columns": [len(spatial_cols), len(canopy_cols), len(elevation_cols), len(density_cols), len(other_cols)],
    "columns": [spatial_cols, canopy_cols, elevation_cols, density_cols, other_cols],
})
column_groups

,group,number_of_columns,columns
0,spatial / identifier,3,"[X, Y, PolygonFID]"
1,canopy,3,"[CanopyCover, GapFraction, LAI]"
2,elevation,46,"[elev_aad_z, elev_canopy_relief_ratio, elev_AI..."
3,density,10,"[density_metrics[0], density_metrics[1], densi..."
4,unclassified,0,[]


In [11]:
schema = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "non_null": df.notna().sum(),
    "missing_pct": df.isna().mean().mul(100),
    "unique": df.nunique(dropna=True),
})
schema

,dtype,non_null,missing_pct,unique
X,float64,71996,0.0,19434
Y,int64,71996,0.0,71996
PolygonFID,float64,71996,0.0,41084
CanopyCover,float64,71996,0.0,41084
GapFraction,float64,71996,0.0,40803
LAI,float64,71996,0.0,42012
elev_aad_z,float64,71996,0.0,40317
elev_canopy_relief_ratio,float64,71996,0.0,36660
elev_AIH_1st,float64,71996,0.0,40355
elev_AIH_5th,float64,71996,0.0,41113


Likely naming conventions (to verify against documentation):

- `elev_percentile_*`: height quantiles in the spatial unit;
- `elev_AIH_*`: quantiles of an accumulated height distribution;
- `elev_mean`, `elev_max`,`elev_sqrt_mean_sq`, `elev_madmedian` etc.: summaries of normalised return elevations;
- `density_metrics[i]`: proportions in height bins or strata. The bin boundaries are not encoded in this file.

## 3. Data-quality audit

In [12]:
missing = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_pct": df.isna().mean().mul(100),
}).sort_values(["missing_pct", "missing_count"], ascending=False)

missing[missing["missing_count"] > 0]

,missing_count,missing_pct
density_metrics[9],71996,100.0


In [13]:
all_missing_cols = df.columns[df.isna().all()].tolist()
constant_cols = [c for c in df.columns if df[c].nunique(dropna=True) == 1]

quality_summary = pd.Series({
    "full duplicate rows": int(df.duplicated().sum()),
    "duplicate X/Y pairs": int(df.duplicated(["X", "Y"]).sum()),
    "duplicate PolygonFID values": int(df["PolygonFID"].duplicated().sum()),
    "all-missing columns": len(all_missing_cols),
    "constant non-missing columns": len(constant_cols),
})
display(quality_summary.to_frame("count"))
print("All-missing columns:", all_missing_cols)
print("Constant columns:", constant_cols)

,count
full duplicate rows,0
duplicate X/Y pairs,0
duplicate PolygonFID values,30912
all-missing columns,1
constant non-missing columns,0


All-missing columns: ['density_metrics[9]']
Constant columns: []


Do not silently drop columns during investigation. An entirely missing final density field may indicate an export convention or failed calculation; that is information worth recording. We create a separate analysis frame only at the end.

## 4. Validate expected ranges and identities

In [ ]:
range_checks = pd.Series({
    "CanopyCover outside [0, 1]": (~df["CanopyCover"].between(0, 1)).sum(),
    "GapFraction outside [0, 1]": (~df["GapFraction"].between(0, 1)).sum(),
    "LAI below 0": (df["LAI"] < 0).sum(),
    "elev_min > elev_mean": (df["elev_min"] > df["elev_mean"]).sum(),
    "elev_mean > elev_max": (df["elev_mean"] > df["elev_max"]).sum(),
})
range_checks.to_frame("violating_rows")

In [ ]:
cover_gap_error = df["CanopyCover"] + df["GapFraction"] - 1
print(f"Maximum |CanopyCover + GapFraction - 1|: {cover_gap_error.abs().max():.2e}")
print(f"Correlation: {df['CanopyCover'].corr(df['GapFraction']):.6f}")

fig, ax = plt.subplots(figsize=(5, 4))
ax.hist(cover_gap_error, bins=40)
ax.set(title="Numerical error in complementary cover metrics", xlabel="CanopyCover + GapFraction - 1", ylabel="Rows")
plt.show()

If the error above is only rounding-scale, `CanopyCover` and `GapFraction` contain the same information in opposite directions. Keep whichever is easiest to interpret in a later model, rather than treating them as independent predictors.

In [ ]:
percentile_order = ["1st", "5th", "10th", "20th", "25th", "30th", "40th", "50th", "60th", "70th", "75th", "80th", "90th", "95th", "99th"]
percentile_cols = [f"elev_percentile_{p}" for p in percentile_order]
percentile_violations = df[percentile_cols].diff(axis=1).iloc[:, 1:].lt(0).any(axis=1)
print(f"Rows with decreasing elevation percentiles: {percentile_violations.sum():,}")

In [ ]:
usable_density_cols = [c for c in density_cols if df[c].notna().any()]
density_sum = df[usable_density_cols].sum(axis=1, min_count=1)
density_check = pd.Series({
    "usable density columns": len(usable_density_cols),
    "minimum row sum": density_sum.min(),
    "median row sum": density_sum.median(),
    "maximum row sum": density_sum.max(),
    "rows summing approximately to 1": np.isclose(density_sum, 1, atol=0.001).sum(),
})
density_check.to_frame("value")

A density sum below 1 is not automatically an error: a ground/low-return category may be omitted, or rows may have no eligible returns. The generating software's definition is needed before interpreting these fields.

## 5. Distributions and zero patterns

In [ ]:
key_cols = ["CanopyCover", "GapFraction", "LAI", "elev_min", "elev_mean", "elev_median_z", "elev_max", "elev_stddev"]
quantiles = [0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
df[key_cols].describe(percentiles=quantiles).T

In [ ]:
axes = df[key_cols].hist(figsize=(14, 8), bins=50, layout=(2, 4))
for ax in axes.ravel():
    ax.set_ylabel("Rows")
plt.suptitle("Distributions of selected interpretable metrics", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
zero_summary = pd.DataFrame({
    "zero_count": df[key_cols].eq(0).sum(),
    "zero_pct": df[key_cols].eq(0).mean().mul(100),
}).sort_values("zero_pct", ascending=False)
zero_summary

In [ ]:
zero_flags = pd.DataFrame({
    "zero_canopy_cover": df["CanopyCover"].eq(0),
    "zero_LAI": df["LAI"].eq(0),
    "zero_mean_height": df["elev_mean"].eq(0),
    "zero_max_height": df["elev_max"].eq(0),
})

zero_pattern_counts = (
    zero_flags.value_counts()
    .rename("rows")
    .reset_index()
    .sort_values("rows", ascending=False)
)
zero_pattern_counts

Zero is a valid numeric value, but its scientific meaning is unresolved. It could represent open ground, no returns above a threshold, a calculation convention, or true zero. Differences between zero patterns are more informative than assuming all zeros mean the same thing.

## 6. Spatial coverage and patterns

In [ ]:
df[["X", "Y", "PolygonFID"]].describe().T

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 6), constrained_layout=True)
for ax, metric, title in zip(
    axes,
    ["CanopyCover", "elev_mean"],
    ["Canopy cover", "Mean elevation metric"],
):
    points = ax.scatter(df["X"], df["Y"], c=df[metric], s=2, cmap="viridis", rasterized=True)
    ax.set(xlabel="X", ylabel="Y", title=title)
    ax.set_aspect("equal")
    fig.colorbar(points, ax=ax, shrink=0.8, label=metric)
plt.show()

These maps help reveal block boundaries, gaps, strips, and spatial clustering of zero or extreme values. The axis units and map projection should not be stated until the coordinate reference system is confirmed.

## 7. Relationships among selected metrics

In [ ]:
relationship_cols = ["CanopyCover", "LAI", "elev_mean", "elev_max", "elev_stddev"]
corr = df[relationship_cols].corr()

fig, ax = plt.subplots(figsize=(6, 5))
image = ax.imshow(corr, vmin=-1, vmax=1, cmap="coolwarm")
ax.set_xticks(range(len(corr)), corr.columns, rotation=45, ha="right")
ax.set_yticks(range(len(corr)), corr.index)
for i in range(len(corr)):
    for j in range(len(corr)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center")
fig.colorbar(image, ax=ax, label="Pearson correlation")
ax.set_title("Selected metric correlations")
plt.tight_layout()
plt.show()

In [ ]:
plot_sample = df.sample(n=min(10_000, len(df)), random_state=42)
fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
axes[0].scatter(plot_sample["CanopyCover"], plot_sample["LAI"], s=3, alpha=0.25)
axes[0].set(xlabel="CanopyCover", ylabel="LAI", title="Cover and LAI")
axes[1].scatter(plot_sample["CanopyCover"], plot_sample["elev_mean"], s=3, alpha=0.25)
axes[1].set(xlabel="CanopyCover", ylabel="elev_mean", title="Cover and mean height metric")
plt.show()

Correlation is descriptive here. Shared zero patterns and mathematical dependence can create strong correlations; they do not establish an ecological causal relationship.

## 8. Explicit analysis copy and investigation record

In [ ]:
# Exclude only fields that contain no observations; retain raw_df as originally loaded.
analysis_df = raw_df.drop(columns=all_missing_cols).copy()

print(f"raw_df shape:      {raw_df.shape}")
print(f"analysis_df shape: {analysis_df.shape}")
print("Excluded from analysis_df:", all_missing_cols)

### What this file establishes after running the notebook

Fill this section from the displayed evidence rather than from assumptions:

- **Observational unit:** Are `X/Y` and `PolygonFID` unique per row?
- **Completeness:** Which fields are missing or entirely empty?
- **Internal consistency:** Do cover/gap, height ordering, percentiles, and density fields behave as expected?
- **Distributions:** Which variables are zero-inflated, skewed, or extreme?
- **Spatial structure:** Are gaps or extremes geographically clustered?

### Questions for the data provider / processing documentation

1. What is the coordinate reference system for `X` and `Y`?
2. What geometry and area does each `PolygonFID` represent?
3. Are elevation values height-normalised metres above ground, or absolute elevations?
4. Which returns and minimum-height thresholds were used for canopy, LAI, elevation, and density metrics?
5. What exactly do `AIH`, `aad_z`, `curt_mean_cube`, `IQ`, and each density index mean?
6. What does a zero mean for each metric, and why is the last density field empty?
7. Were the same processing settings used for every year and block?

Those answers are prerequisites for defensible cross-year comparisons.